# SSAFY AI Challenge - Improved VQA Baseline

## 개선 사항
- **Step 1**: 모델 변경 → `openbmb/MiniCPM-V-2` (2.8B Vision-Language Model)
- **Step 2**: 전체 train 데이터 사용 (200개 샘플링 제거)
- **Step 3**: dev 데이터를 활용한 데이터 증강 (응답1~5 majority voting으로 pseudo-label 생성)
- **Step 4**: 프롬프트 개선 (한국어 특화, 구조화된 지시사항)
- **Step 5**: Epoch 증가 (1 → 3)
- **Step 6**: 하이퍼파라미터 조정 (lr, LoRA rank, scheduler 등)
- **Step 7**: YOLO 객체 탐지 결과를 VLM 프롬프트에 주입 (Knowledge Distillation)

실행 환경: **Google Colab Pro (A100 40GB)**

# 0. 환경 준비

In [ ]:
!pip -q install transformers>=4.40.0 accelerate
!pip -q install peft>=0.13.2 bitsandbytes>=0.43.0 datasets pillow pandas tqdm
!pip -q install ultralytics timm sentencepiece protobuf hf_xet
!pip -q install qwen-vl-utils
# 설치 완료 후 런타임 > 세션 다시 시작


In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

# 1. 데이터 준비 (Google Drive 마운트 및 압축 해제)

In [ ]:
import os, glob, zipfile

DATA_DIR = '/content'

zip_files = glob.glob('/content/*.zip')
if zip_files:
    zip_path = zip_files[0]
    print(f'zip 발견: {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('압축 해제 완료')
else:
    print('zip 없음 - 이미 압축 해제된 상태로 진행')


In [ ]:
for f in ['train.csv', 'test.csv', 'dev.csv', 'sample_submission.csv']:
    p = os.path.join(DATA_DIR, f)
    print(f'{f}: {"OK" if os.path.exists(p) else "NOT FOUND"}')
for d in ['train', 'test', 'dev']:
    p = os.path.join(DATA_DIR, d)
    if os.path.isdir(p):
        print(f'{d}/: OK ({len(os.listdir(p))} files)')
    else:
        print(f'{d}/: NOT FOUND')


# 2. 라이브러리 임포트 및 전역 설정

In [ ]:
import os, re, math, random, json
from collections import Counter
from dataclasses import dataclass
from typing import Any, List, Dict, Optional

import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None

# ── 전역 설정 ──────────────────────────────────────────────────
SEED        = 42
MODEL_ID       = 'Qwen/Qwen2.5-VL-7B-Instruct'  # Step 1: 변경
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# 학습 하이퍼파라미터 (Step 5, 6)
NUM_EPOCHS      = 3          # Step 5: 1→3
BATCH_SIZE      = 4          # A100 40GB 기준
GRAD_ACCUM      = 4
LR              = 5e-5       # Step 6: 1e-4→5e-5 (안정적 수렴)
WARMUP_RATIO    = 0.05
MAX_NEW_TOKENS  = 8
LORA_R          = 16         # Step 6: 8→16 (표현력 증가)
LORA_ALPHA      = 32         # Step 6: 16→32
LORA_DROPOUT    = 0.05
WEIGHT_DECAY    = 0.01       # Step 6: AdamW weight decay 추가
MAX_GRAD_NORM   = 1.0        # Step 6: gradient clipping

# Step 7: YOLO 사용 여부
USE_YOLO        = True
YOLO_CONF       = 0.25       # 탐지 신뢰도 임계값
YOLO_MAX_OBJ    = 5          # 프롬프트에 포함할 최대 객체 수

# 경로
DATA_DIR    = '/content'
SAVE_DIR    = '/content/qwen25vl_7b_lora'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | Model: {MODEL_ID}")
print(f"Epochs: {NUM_EPOCHS} | LR: {LR} | LoRA r: {LORA_R}")

# 3. Step 7 - YOLO 객체 탐지 모듈

YOLOv8n으로 이미지에서 객체를 탐지하고, 탐지 결과를 자연어로 변환하여 VLM 프롬프트에 주입합니다.  
이를 통해 탐지 모델의 spatial knowledge를 VLM에 distill합니다.

In [ ]:
import subprocess, sys

yolo_model = None

if USE_YOLO:
    try:
        from ultralytics import YOLO
    except ImportError:
        print('ultralytics 설치 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics', '-q'])
        from ultralytics import YOLO
    yolo_model = YOLO('yolov8n.pt')
    print('YOLO loaded')


def detect_objects(img):
    if yolo_model is None:
        return ''
    try:
        res    = yolo_model(img, conf=YOLO_CONF, verbose=False)[0]
        boxes  = res.boxes
        if boxes is None or len(boxes) == 0:
            return ''
        confs  = boxes.conf.cpu().numpy()
        clsids = boxes.cls.cpu().numpy().astype(int)
        names  = res.names
        order  = np.argsort(confs)[::-1][:YOLO_MAX_OBJ]
        objs   = [f'{names[clsids[i]]}({confs[i]:.2f})' for i in order]
        return '[detected: ' + ', '.join(objs) + ']'
    except Exception:
        return ''


# 4. dev 데이터 - Soft Label 학습

dev의 answer1~5를 응답 비율로 변환해 soft label로 사용합니다.

- **train**: 정답 있음 → CrossEntropy (hard label)
- **dev**: answer1~5 비율 → KL Divergence (soft label)


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, 'dev.csv'))

print(f'Train: {len(train_df)}개 | Test: {len(test_df)}개 | Dev: {len(dev_df)}개')
print('Dev 컬럼:', dev_df.columns.tolist())


In [ ]:
# dev answer1~5 -> soft label (응답 비율) 생성
LABEL2ID = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
vote_cols = [c for c in dev_df.columns if c.startswith('answer') and c != 'answer']
print(f'응답 컬럼: {vote_cols}')

def make_soft_label(row, vote_cols):
    """answer1~5 응답 비율을 [a,b,c,d] 확률 벡터로 변환."""
    votes = [str(row[c]).strip().lower() for c in vote_cols if pd.notna(row[c])]
    valid = [v for v in votes if v in LABEL2ID]
    if not valid:
        return None
    dist = [0.0, 0.0, 0.0, 0.0]
    for v in valid:
        dist[LABEL2ID[v]] += 1.0
    total = sum(dist)
    return [d / total for d in dist]  # [p_a, p_b, p_c, p_d]

dev_df['soft_label'] = dev_df.apply(lambda r: make_soft_label(r, vote_cols), axis=1)
dev_soft = dev_df.dropna(subset=['soft_label']).copy()

print(f'Dev soft-label 샘플: {len(dev_soft)}/{len(dev_df)}')
print('예시:', dev_soft['soft_label'].iloc[0])


In [ ]:
# Step 2: train 전체 + dev soft-label 샘플 통합
keep_cols = ['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer']

# train: hard label 샘플 (soft_label=None)
train_hard = train_df[keep_cols].copy()
train_hard['soft_label'] = None
train_hard['use_soft']   = False

# dev: soft label 샘플
dev_soft_df = dev_soft[['id','path','question','a','b','c','d','soft_label']].copy()
dev_soft_df['answer']   = None
dev_soft_df['use_soft'] = True

combined_df = pd.concat([train_hard, dev_soft_df], ignore_index=True
              ).sample(frac=1, random_state=SEED).reset_index(drop=True)

split        = int(len(combined_df) * 0.9)
train_subset = combined_df.iloc[:split].reset_index(drop=True)
valid_subset = combined_df.iloc[split:].reset_index(drop=True)

print(f'학습: {len(train_subset)}개 (hard: {(~train_subset["use_soft"]).sum()} '
      f'| soft: {train_subset["use_soft"].sum()})')
print(f'검증: {len(valid_subset)}개')


# 5. 모델 & 토크나이저 로드 (MiniCPM-V-2)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,  # A100 네이티브
)

print('Processor 로드 중...')
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*28*28,
    max_pixels=384*28*28,
)

print('모델 로드 중 (4-bit)...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()
print('모델 로드 완료')


In [ ]:
target_modules = list({
    name.split('.')[-1]
    for name, _ in base_model.named_modules()
    if any(k in name for k in ['q_proj','k_proj','v_proj','o_proj',
                                'gate_proj','up_proj','down_proj'])
}) or ['q_proj', 'v_proj']
print('LoRA 적용 모듈:', target_modules)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', target_modules=target_modules, task_type='CAUSAL_LM',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


# 6. Step 4 - 개선된 프롬프트 템플릿

In [ ]:
SYSTEM_INSTRUCT = (
    '당신은 이미지를 분석하여 4지선다 문제를 푸는 전문 AI 어시스턴트입니다. '
    '이미지를 주의 깊게 관찰하고, 질문과 선지를 바탕으로 가장 적합한 답을 선택하세요. '
    '반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요. 설명이나 다른 텍스트는 출력하지 마세요.'
)

def build_mc_prompt(question, a, b, c, d, yolo_info=''):
    prefix = f'{yolo_info}\n' if yolo_info else ''
    return (
        f'{prefix}'
        f'질문: {question}\n'
        f'(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n'
        '정답 (a/b/c/d 중 하나):'
    )


# 7. Dataset & DataLoader

In [ ]:
class VQAMCDataset(Dataset):
    """
    Qwen2.5-VL processor 기반 데이터셋.
    - use_soft=False: train hard label -> CrossEntropy
    - use_soft=True : dev soft label  -> KL Divergence
    """
    def __init__(self, df, processor, use_yolo=False):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.use_yolo  = use_yolo

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row      = self.df.iloc[i]
        img_path = os.path.join(DATA_DIR, row['path']) if not os.path.isabs(row['path']) else row['path']
        img      = Image.open(img_path).convert('RGB')
        yolo_info = detect_objects(img) if self.use_yolo else ''
        user_text = build_mc_prompt(
            str(row['question']),
            str(row['a']), str(row['b']), str(row['c']), str(row['d']),
            yolo_info=yolo_info,
        )
        use_soft   = bool(row['use_soft'])
        answer     = str(row['answer']).strip().lower() if not use_soft else None
        soft_label = list(row['soft_label']) if use_soft else None
        return {
            'image': img, 'user_text': user_text,
            'answer': answer, 'soft_label': soft_label, 'use_soft': use_soft,
        }


@dataclass
class DataCollator:
    processor: Any

    def __call__(self, batch):
        texts, images_list = [], []
        ids_list, lbl_list, soft_list, use_soft_list = [], [], [], []

        for s in batch:
            img, ut, ans, use_soft = s['image'], s['user_text'], s['answer'], s['use_soft']

            # Qwen2.5-VL messages 포맷
            msgs = [
                {
                    'role': 'system',
                    'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': img},
                        {'type': 'text', 'text': ut},
                    ]
                }
            ]
            if not use_soft and ans:
                msgs.append({'role': 'assistant', 'content': [{'type': 'text', 'text': ans}]})

            text = self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images_list.append([img])
            soft_list.append(s['soft_label'] if use_soft else None)
            use_soft_list.append(use_soft)

        # processor로 배치 인코딩 (이미지 + 텍스트 통합)
        enc = self.processor(
            text=texts,
            images=images_list,
            padding=True,
            return_tensors='pt',
        )

        # labels: hard label만 answer 위치 unmask, soft/없으면 전체 -100
        labels = enc['input_ids'].clone()
        for j, (s, text) in enumerate(zip(batch, texts)):
            if not s['use_soft'] and s['answer']:
                # prompt 부분 마스킹: assistant 응답 이전까지 -100
                msgs_no_ans = [
                    {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
                    {'role': 'user', 'content': [
                        {'type': 'image', 'image': s['image']},
                        {'type': 'text', 'text': s['user_text']},
                    ]}
                ]
                prompt_text = self.processor.apply_chat_template(
                    msgs_no_ans, tokenize=False, add_generation_prompt=True
                )
                prompt_enc = self.processor(
                    text=[prompt_text], images=[[s['image']]],
                    return_tensors='pt', padding=False,
                )
                prompt_len = prompt_enc['input_ids'].shape[1]
                labels[j, :prompt_len] = -100
            else:
                labels[j, :] = -100

        enc['labels']      = labels
        enc['soft_labels'] = soft_list
        enc['use_soft']    = use_soft_list
        enc['images']      = [s['image'] for s in batch]
        return enc


train_ds = VQAMCDataset(train_subset, processor, use_yolo=USE_YOLO)
valid_ds = VQAMCDataset(valid_subset, processor, use_yolo=USE_YOLO)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor), num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor), num_workers=2, pin_memory=True)
print(f'train: {len(train_loader)} | valid: {len(valid_loader)} batches')


# 8. Step 5+6 - 학습 (3 epochs, 개선된 하이퍼파라미터)

In [ ]:
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999), eps=1e-8,
)
total_steps  = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler('cuda', enabled=True)

kl_loss_fn  = torch.nn.KLDivLoss(reduction='batchmean')
SOFT_WEIGHT = 0.3

CHOICE_TOKEN_IDS = [
    processor.tokenizer.convert_tokens_to_ids('a'),
    processor.tokenizer.convert_tokens_to_ids('b'),
    processor.tokenizer.convert_tokens_to_ids('c'),
    processor.tokenizer.convert_tokens_to_ids('d'),
]
print(f'total: {total_steps} | warmup: {warmup_steps}')
print(f'choice token ids: {CHOICE_TOKEN_IDS}')

best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running, last_avg = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [train]')

    for step, batch in enumerate(pbar, 1):
        soft_labels  = batch.pop('soft_labels')
        use_soft_vec = batch.pop('use_soft')
        batch.pop('images')
        batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs   = model(**batch)
            hard_loss = outputs.loss

            soft_loss    = torch.tensor(0.0, device=DEVICE)
            soft_indices = [j for j, us in enumerate(use_soft_vec) if us]
            if soft_indices and outputs.logits is not None:
                logits = outputs.logits
                for j in soft_indices:
                    last_logit = logits[j, -1, CHOICE_TOKEN_IDS]
                    log_prob   = torch.nn.functional.log_softmax(last_logit.float(), dim=-1)
                    target     = torch.tensor(soft_labels[j], dtype=torch.float, device=DEVICE)
                    soft_loss  = soft_loss + kl_loss_fn(log_prob.unsqueeze(0), target.unsqueeze(0))
                soft_loss = soft_loss / len(soft_indices)

            loss = ((1 - SOFT_WEIGHT) * hard_loss + SOFT_WEIGHT * soft_loss) / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            last_avg = running / GRAD_ACCUM
            pbar.set_postfix({'loss': f'{last_avg:.3f}',
                              'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
            running = 0.0

    history['train_loss'].append(last_avg)

    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [valid]'):
            vb.pop('soft_labels'); vb.pop('use_soft'); vb.pop('images')
            vb = {k: v.to(DEVICE) for k, v in vb.items() if isinstance(v, torch.Tensor)}
            val_loss += model(**vb).loss.item(); val_steps += 1
    avg_val = val_loss / val_steps
    history['val_loss'].append(avg_val)
    print(f'[Epoch {epoch+1}] train={last_avg:.4f} | val={avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(SAVE_DIR + '_best')
        processor.save_pretrained(SAVE_DIR + '_best')
        print(f'  -> best saved (val={best_val_loss:.4f})')
    model.train()

model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print('done | history:', history)


# 9. 추론 (Inference)

In [ ]:
from peft import PeftModel

def extract_choice(text):
    text = text.strip().lower()
    for line in reversed(text.splitlines()):
        line = line.strip()
        if line in ('a','b','c','d'): return line
        m = re.search(r'[\(\[]*([abcd])[\)\]]*', line)
        if m: return m.group(1)
    m = re.search(r'\b([abcd])\b', text)
    return m.group(1) if m else 'a'

print('Best 모델 로드 중...')
infer_base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
infer_model = PeftModel.from_pretrained(infer_base, SAVE_DIR + '_best')
infer_model.eval()
print('추론 모델 로드 완료')


In [ ]:
preds = []

for i in tqdm(range(len(test_df)), desc='Inference'):
    row      = test_df.iloc[i]
    img_path = os.path.join(DATA_DIR, row['path']) if not os.path.isabs(row['path']) else row['path']
    img      = Image.open(img_path).convert('RGB')
    yolo_info = detect_objects(img) if USE_YOLO else ''
    user_text = build_mc_prompt(
        str(row['question']),
        str(row['a']), str(row['b']), str(row['c']), str(row['d']),
        yolo_info=yolo_info,
    )
    msgs = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': user_text},
        ]}
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[[img]],
        return_tensors='pt', padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = infer_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    # 새로 생성된 토큰만 디코딩
    new_ids = out_ids[:, inputs['input_ids'].shape[1]:]
    response = processor.batch_decode(new_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(response))

submission = pd.DataFrame({'id': test_df['id'], 'answer': preds})
submission.to_csv('/content/submission.csv', index=False)
print('submission.csv 저장 완료')
print('정답 분포:', submission['answer'].value_counts().to_dict())


# 10. (선택) 검증셋 정확도 확인

In [ ]:
val_preds, val_golds = [], []

for i in tqdm(range(len(valid_subset)), desc='Val Accuracy'):
    row      = valid_subset.iloc[i]
    img_path = os.path.join(DATA_DIR, row['path']) if not os.path.isabs(row['path']) else row['path']
    img      = Image.open(img_path).convert('RGB')
    yolo_info = detect_objects(img) if USE_YOLO else ''
    user_text = build_mc_prompt(
        str(row['question']),
        str(row['a']), str(row['b']), str(row['c']), str(row['d']),
        yolo_info=yolo_info,
    )
    msgs = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_INSTRUCT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': user_text},
        ]}
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[[img]],
        return_tensors='pt', padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = infer_model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, eos_token_id=processor.tokenizer.eos_token_id,
        )
    new_ids  = out_ids[:, inputs['input_ids'].shape[1]:]
    response = processor.batch_decode(new_ids, skip_special_tokens=True)[0]
    val_preds.append(extract_choice(response))
    val_golds.append(str(row['answer']).strip().lower())

acc = sum(p == g for p, g in zip(val_preds, val_golds)) / len(val_golds)
print(f'Val Accuracy: {acc*100:.2f}% ({sum(p==g for p,g in zip(val_preds,val_golds))}/{len(val_golds)})')
